# Etapa 05 — Bronze incremental V2

Leia primeiro. A Bronze preserva JSON de origem e metadados sem corrigir registros inválidos. Processa **um lote por execução** para demonstrar incrementalidade, idempotência e chegada tardia.

Na pasta Git use `chore/estrutura-inicial`, faça Pull e, se a sessão Python tiver importado módulos antigos, reinicie Python antes de executar. Sem instalação adicional. Não execute os lotes em paralelo.

In [0]:
# CMD 1 — Definir widgets. Selecione o lote antes de prosseguir.
dbutils.widgets.text('catalogo', spark.catalog.currentCatalog())
dbutils.widgets.text('data_referencia', '2026-01-15')
dbutils.widgets.dropdown('lote', '01_inicial', ['01_inicial','02_incremental','03_atrasado'])
dbutils.widgets.text('batch_id','')
print('Execute separadamente 01_inicial, 02_incremental, 03_atrasado. Para testar idempotência, repita 01_inicial após a primeira execução.')

Execute separadamente 01_inicial, 02_incremental, 03_atrasado. Para testar idempotência, repita 01_inicial após a primeira execução.


In [0]:
# CMD 2 — Resolver o pacote da pasta Git
from pathlib import Path
import sys
atual=Path.cwd().resolve()
raiz=next((p for p in [atual,*atual.parents] if (p/'src'/'medallion').is_dir() and (p/'pyproject.toml').is_file()),None)
if raiz is None:
    raise RuntimeError('Abra o notebook dentro da pasta Git do projeto')
if str(raiz/'src') not in sys.path:
    sys.path.insert(0,str(raiz/'src'))
from medallion.config import PipelineConfig
from medallion.bronze.ingest_v2 import executar_bronze_v2, preparar_lote

In [0]:
# CMD 3 — Preparar execução. Confirme lote e contagens antes de gravar.
import uuid
parametros={'ambiente':'dev','modo':'incremental','catalogo':dbutils.widgets.get('catalogo').strip(),'data_referencia':dbutils.widgets.get('data_referencia').strip()}
identificador=dbutils.widgets.get('batch_id').strip()
if identificador:
    parametros['batch_id']=identificador
config=PipelineConfig(**parametros)
lote=dbutils.widgets.get('lote').strip()
itens,manifesto_hash=preparar_lote(config.caminho_entrada,config.data_referencia,lote)
print('Lote:',lote,'Arquivos:',len(itens),'Registros:',sum(len(linhas) for _,linhas in itens))
print('Manifesto SHA256:',manifesto_hash)

Lote: 03_atrasado Arquivos: 19 Registros: 1520
Manifesto SHA256: 480bcccda11f61d9e29f6a9dd56a0f106b7fd05ad32903b0c768428e3e7113a9


In [0]:
# CMD 4 — Ingerir o lote via MERGE idempotente; NÃO execute em paralelo.
resultado_bronze=executar_bronze_v2(spark,config,lote,str(uuid.uuid4()))

BRONZE V2: OK {"arquivos_processados": 19, "lote": "03_atrasado", "manifesto_sha256": "480bcccda11f61d9e29f6a9dd56a0f106b7fd05ad32903b0c768428e3e7113a9", "registros_por_fonte": {"cartoes_cdc": 0, "clientes_cdc": 1, "contas_cdc": 0, "estornos": 1, "eventos_risco": 0, "transacoes": 1518}, "run_id": "cf5f9453-7a4c-47f9-a699-3665fc9783c1"}


In [0]:
# CMD 5 — Inspecionar progresso da Bronze e controle de arquivos.
controle=config.tabela_sql('controle','arquivos_bronze_v2')
display(spark.sql(f"SELECT lote, fonte, count(*) arquivos, sum(registros) linhas_arquivo FROM {controle} WHERE data_referencia='{config.data_referencia}' GROUP BY lote,fonte ORDER BY lote,fonte"))
for fonte in ('clientes_cdc','contas_cdc','cartoes_cdc','transacoes','eventos_risco','estornos'):
    nome=config.tabela_sql('bronze','raw_'+fonte)
    print(fonte,[(r['lote'],r['qtd']) for r in spark.sql(f"SELECT lote,count(*) qtd FROM {nome} WHERE data_referencia='{config.data_referencia}' GROUP BY lote ORDER BY lote").collect()])

lote,fonte,arquivos,linhas_arquivo
01_inicial,cartoes_cdc,1,22
01_inicial,clientes_cdc,1,22
01_inicial,contas_cdc,1,22
01_inicial,estornos,1,1
01_inicial,eventos_risco,1,1
01_inicial,transacoes,60,6080
02_incremental,cartoes_cdc,1,2
02_incremental,clientes_cdc,1,1
02_incremental,contas_cdc,1,1
02_incremental,estornos,1,2


clientes_cdc [('01_inicial', 22), ('02_incremental', 1), ('03_atrasado', 1)]
contas_cdc [('01_inicial', 22), ('02_incremental', 1)]
cartoes_cdc [('01_inicial', 22), ('02_incremental', 2)]
transacoes [('01_inicial', 6080), ('02_incremental', 4560), ('03_atrasado', 1518)]
eventos_risco [('01_inicial', 1), ('02_incremental', 3)]
estornos [('01_inicial', 1), ('02_incremental', 2), ('03_atrasado', 1)]


## Critérios de aceite e envio

1. Execute **01_inicial**, salve outputs, execute novamente 01_inicial e confirme que as contagens não aumentaram.
2. Execute **02_incremental** e **03_atrasado**, sempre separadamente. Compare as contagens de transações: 6080, 4560, 1518 (total 12158).
3. Confirme seis tabelas `raw_*` e `medallion_dev_controle.arquivos_bronze_v2`. Os registros CDC duplicados e transações recusadas devem ser preservados sem filtragem nesta etapa.
4. Exporte este IPYNB **com outputs** e envie para revisão. Não efetue merge em develop/main. Falhas devem interromper, sem apagar arquivos ou tabelas. A validação Silver ocorrerá depois.